In [7]:
import numpy as np
import math

batch_size = 2 #Two sentences at once
seq_len = 3
d_model = 8
num_heads = 2
d_k = d_model // num_heads # 4

# Our input X (batch_size, seq_len, d_model) -> (2, 3, 8)
X = np.random.randn(batch_size, seq_len, d_model)

# The massive weight matrices (d_model, d_model) -> (8, 8)
# Notice they project d_model to d_model, not d_model to d_k!
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model) # The final output projection (Section 3.2)

In [8]:
print(X)
print("\n--\n")
print(W_Q)

[[[ 3.12663375  0.03496998  0.69826796  1.6615428  -0.32687837
    0.97937467  0.37553962 -1.16464687]
  [ 0.65709962  0.24518182  0.50023725 -1.04120742  0.0796048
   -0.33610892  1.08503029 -0.13992652]
  [-0.81005901 -4.72276323 -1.25228787 -2.0209983  -0.60066259
    1.53528988 -0.47515711 -0.12994589]]

 [[ 0.76074508  0.35077634  0.63628082  0.4198253   0.1557314
   -0.53317889  0.66509279  0.26534629]
  [-0.02050731 -1.29661761  1.13339257 -0.86987087  0.45955621
   -0.28852242  1.53082568  0.15396253]
  [-0.08848693 -1.54142062 -0.76243609  2.38695199  0.29293722
    0.93797908 -0.21954165 -1.51807374]]]

--

[[ 0.18127555  0.01976633  0.52482637  0.75757654 -0.88558449  0.15914937
   0.60370255 -0.27750852]
 [ 0.74517289 -0.78867035 -0.335413    0.99461743  2.61461152 -1.16087447
   1.26423969  1.32935255]
 [ 1.03221271  0.83824214  0.05339044  0.82091949 -0.16246653 -0.54104075
  -0.63808197  0.9709649 ]
 [-1.52183412 -0.40417948  0.76347571 -0.83122447  0.74115115  0.3601523

In [52]:
# Important - Project entire X to Q,K,V before 
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(Q.shape)

Q_reshaped = Q.reshape(batch_size, seq_len, num_heads, d_k)

print("Reshaped")
print(Q_reshaped.shape)

Q_reshaped_corr = Q_reshaped.transpose(0,2,1,3)
print("Transpose - to get num_heads before seq_len and d_k")
print(Q_reshaped_corr.shape)

## Do the others
K_reshaped = K.reshape(batch_size, seq_len, num_heads, d_k)
K_reshaped_corr = K_reshaped.transpose(0,2,1,3)
V_reshaped = V.reshape(batch_size, seq_len, num_heads, d_k)
V_reshaped_corr = V_reshaped.transpose(0,2,1,3)

S = Q_reshaped_corr @ K_reshaped_corr.transpose(0,1,3,2)
S = S / (math.sqrt(d_k))

print(f"\nS is \n {S}\n and shape {S.shape}")

## Applying Softmax

S_max_subtracted = S - np.max(S, axis=-1, keepdims=True)
S_max_subtracted_epow = np.exp(S_max_subtracted)
S_max_subtracted_pow_sum = np.sum(S_max_subtracted_epow, axis=-1, keepdims=True)

A = S_max_subtracted_epow/S_max_subtracted_pow_sum
print(f"\nA is \n {A}\n and shape {A.shape}")

output = A @ V_reshaped_corr
print(f"\noutput is \n {output}\n and shape {output.shape}")

output_transpose = output.transpose(0,2,1,3)
output_reshaped = output_transpose.reshape(batch_size, seq_len, d_model)
print(f"\noutput_reshaped is \n {output_reshaped}\n and shape {output_reshaped.shape}")

## Final Project

Y = output_reshaped @ W_O


(2, 3, 8)
Reshaped
(2, 3, 2, 4)
Transpose - to get num_heads before seq_len and d_k
(2, 2, 3, 4)

S is 
 [[[[ 18.5168762    7.34397586 -18.8609567 ]
   [  4.85036879  14.15010968  -3.13174038]
   [-17.7744724   -6.38936603  28.06195204]]

  [[-16.24585036   1.62781388   7.83273943]
   [ -1.25620271  -0.85764465   0.42874079]
   [-47.71283284  -6.94885532  46.04200723]]]


 [[[  1.46458102   2.56409902  -3.33999771]
   [  0.19703979  10.95368107 -11.91808634]
   [ -6.49655709 -10.49756421  12.22154917]]

  [[ -0.32448536  -0.56375108   3.18732954]
   [ -0.7092806    2.25135097  -7.47168295]
   [  5.83480322  11.43990739 -26.04482972]]]]
 and shape (2, 2, 3, 3)

A is 
 [[[[9.99985950e-01 1.40496315e-05 5.84799949e-17]
   [9.14395591e-05 9.99908529e-01 3.12282864e-08]
   [1.24020591e-20 1.09139678e-15 1.00000000e+00]]

  [[3.48277299e-11 2.01538915e-03 9.97984611e-01]
   [1.26874227e-01 1.89001380e-01 6.84124393e-01]
   [1.91774255e-41 9.69082616e-24 1.00000000e+00]]]


 [[[2.49319947e-01